In [69]:
!pip install spacy

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [70]:
import spacy
from spacy.training.example import Example

# Example CTI training data (you should expand this dataset)
TRAIN_DATA = [
    ("The malware TrickBot contacted 192.168.1.5",
     {"entities": [(12, 20, "MALWARE"), (31, 42, "IP")]}),

    ("APT29 used spear phishing with the domain evil.com",
     {"entities": [(0, 5, "THREAT_ACTOR"), (42, 50, "DOMAIN")]}),

    ("CVE-2021-44228 was exploited in the attack",
     {"entities": [(0, 15, "CVE")]}),
]


In [71]:
# Load base model
nlp = spacy.load("en_core_web_sm")

# Add new NER pipe if not exists
if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

# Add new labels
labels = ["MALWARE", "IP", "DOMAIN", "CVE", "THREAT_ACTOR"]
for label in labels:
    ner.add_label(label)

# Disable other pipes during training
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.resume_training()
    for epoch in range(30):
        losses = {}
        for text, annotations in TRAIN_DATA:
            example = Example.from_dict(nlp.make_doc(text), annotations)
            nlp.update([example], drop=0.3, losses=losses)
        print(f"Epoch {epoch} Losses: {losses}")


/usr/local/lib/python3.12/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "CVE-2021-44228 was exploited in the attack" with entities "[(0, 15, 'CVE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


Epoch 0 Losses: {'ner': np.float32(7.8488536)}
Epoch 1 Losses: {'ner': np.float32(5.814598)}
Epoch 2 Losses: {'ner': np.float32(5.983693)}
Epoch 3 Losses: {'ner': np.float32(6.766383)}
Epoch 4 Losses: {'ner': np.float32(5.465641)}
Epoch 5 Losses: {'ner': np.float32(4.602218)}
Epoch 6 Losses: {'ner': np.float32(5.952222)}
Epoch 7 Losses: {'ner': np.float32(4.6427355)}
Epoch 8 Losses: {'ner': np.float32(5.291369)}
Epoch 9 Losses: {'ner': np.float32(3.8079195)}
Epoch 10 Losses: {'ner': np.float32(3.8357391)}
Epoch 11 Losses: {'ner': np.float32(3.8974755)}
Epoch 12 Losses: {'ner': np.float32(3.6796682)}
Epoch 13 Losses: {'ner': np.float32(2.7048247)}
Epoch 14 Losses: {'ner': np.float32(2.35602)}
Epoch 15 Losses: {'ner': np.float32(2.1273415)}
Epoch 16 Losses: {'ner': np.float32(1.9990624)}
Epoch 17 Losses: {'ner': np.float32(1.5405682)}
Epoch 18 Losses: {'ner': np.float32(1.206914)}
Epoch 19 Losses: {'ner': np.float32(1.091989)}
Epoch 20 Losses: {'ner': np.float32(0.5214691)}
Epoch 21 Loss

In [72]:
test_text = "Indicators show TrickBot linked to 8.8.8.8 and CVE-2019-1234 exploited."
doc = nlp(test_text)

print("Entities Found:")
for ent in doc.ents:
    print(ent.text, ent.label_)


Entities Found:
TrickBot MALWARE
8.8.8.8 IP


In [73]:
nlp.to_disk("./cti_ner_model")
print("Model saved to ./cti_ner_model")


Model saved to ./cti_ner_model


In [74]:
nlp = spacy.load("./cti_ner_model")

In [75]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text


In [77]:
def extract_cti_entities(pdf_path):
    text = extract_text_from_pdf(pdf_path)
    doc = nlp(text)

    results = []
    for ent in doc.ents:
        results.append({"entity": ent.text, "label": ent.label_})
    return results


pdf_file = "/content/Aperture Labs Report.pdf"
entities = extract_cti_entities(pdf_file)

print("Extracted CTI Entities:")
for e in entities:
    print(e)


Extracted CTI Entities:
{'entity': '2', 'label': 'IP'}
{'entity': 'July 20 , 2025', 'label': 'DATE'}
{'entity': 'Januar', 'label': 'DOMAIN'}
{'entity': '1, 2025', 'label': 'DATE'}
{'entity': 'June 30 , 2025', 'label': 'DATE'}
{'entity': 'Summar', 'label': 'MALWARE'}
{'entity': 'Thr', 'label': 'PERSON'}
{'entity': 'Int', 'label': 'MALWARE'}
{'entity': 'F ocus Ar e as & Obser v ed Camp', 'label': 'PRODUCT'}
{'entity': 'Linux Syst', 'label': 'PERSON'}
{'entity': 'Intrusion', 'label': 'DOMAIN'}
{'entity': '2025\ue082', 'label': 'DOMAIN'}
{'entity': 'Aper', 'label': 'THREAT_ACTOR'}
{'entity': 'Unp', 'label': 'THREAT_ACTOR'}
{'entity': 'Librar', 'label': 'THREAT_ACTOR'}
{'entity': 'Insuf', 'label': 'NORP'}
{'entity': 'Access Management', 'label': 'ORG'}
{'entity': 'Str', 'label': 'THREAT_ACTOR'}
{'entity': 'Harden', 'label': 'GPE'}
{'entity': 'Zer o Trust Pr', 'label': 'ORG'}
{'entity': 'Det', 'label': 'PERSON'}
{'entity': 'Cap', 'label': 'PERSON'}
{'entity': 'Appendix', 'label': 'PERSON'}
{